# Melanoma Classification Using the Composite DermoscopyDataset

This notebook walks through the full PyHealth pipeline for binary melanoma classification using dermoscopic images.
It combines three publicly available datasets — **ISIC 2018**, **HAM10000**, and **PH2** — into a single composite
dataset via the `DermoscopyDataset` class.

A key feature of this integration is **mode-based image processing** using the `DermoscopyImageProcessor`.
Instead of always passing the raw image to the model, you can choose to feed:
- `whole` — the full dermoscopic image
- `lesion` — only the lesion region (background zeroed out using the segmentation mask)
- `background` — only the surrounding skin (lesion zeroed out)

This is useful for understanding whether a model is relying on the actual lesion or on background artifacts
(e.g., rulers, gel bubbles, dark corners), which is a key question in the paper this repo accompanies.

---

**Pipeline overview:**
1. Install PyHealth
2. Load the composite dataset
3. Visualize the 3 processing modes
4. Set up task + dataloader
5. Define ResNet50 model
6. Train model
7. Evaluate model
8. Compare modes side-by-side

## Step 0: Install PyHealth

In [ ]:
%pip install pyhealth ipywidgets torchvision

## Step 1: Configure Data Paths

Set the paths to your dataset root and metadata CSV files.

**Expected directory structure under `ROOT`:**
```
ROOT/
├── isic2018/
│   ├── images/          # ISIC_*.JPG
│   └── masks/           # ISIC_*_segmentation.png
├── ham10000/
│   ├── images/          # ISIC_*.jpg
│   └── masks/           # ISIC_*_segmentation.png
└── ph2/
    ├── IMD001/
    │   ├── IMD001_Dermoscopic_Image/
    │   │   └── IMD001.bmp
    │   └── IMD001_lesion/
    │       └── IMD001_lesion.bmp
    └── ...
```

**Metadata CSV requirements:**
- ISIC 2018: columns `image` (filename, e.g. `ISIC_0000001.JPG`) and `label` (0=benign, 1=melanoma)
- HAM10000: columns `image_id` (base name, e.g. `ISIC_0024306`) and `label`
- PH2: columns `Name` (folder name, e.g. `IMD001`) and `label`

You can supply any subset of the three datasets — omit a metadata path to skip that dataset.

In [ ]:
# ── Edit these paths to point to your data ────────────────────────────────────
ROOT = "/path/to/dermoscopy_data"          # root containing isic2018/, ham10000/, ph2/

ISIC2018_METADATA  = "/path/to/isic_bias.csv"       # has 'image' and 'label' columns
HAM10000_METADATA  = "/path/to/ham10000_meta.csv"   # has 'image_id' and 'label' columns
PH2_METADATA       = "/path/to/ph2_meta.csv"        # has 'Name' and 'label' columns
# ─────────────────────────────────────────────────────────────────────────────

# Processing mode: one of "whole", "lesion", "background"
MODE = "whole"

# Training hyperparameters
BATCH_SIZE  = 32
NUM_EPOCHS  = 10
LEARNING_RATE = 1e-4

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Processing mode: {MODE}")

## Step 2: Load the Composite Dataset

In [ ]:
from pyhealth.datasets import DermoscopyDataset

base_dataset = DermoscopyDataset(
    root=ROOT,
    isic2018_metadata_path=ISIC2018_METADATA,
    ham10000_metadata_path=HAM10000_METADATA,
    ph2_metadata_path=PH2_METADATA,
    # To use a subset of datasets, pass e.g. datasets=["isic2018", "ph2"]
)

base_dataset.stats()

### Inspect the unified metadata

The dataset combines all three sources into a single CSV with a `source_dataset` column.

In [ ]:
import pandas as pd
import os

meta = pd.read_csv(os.path.join(ROOT, "dermoscopy-metadata-pyhealth.csv"))
print("Metadata shape:", meta.shape)
print("\nSource distribution:")
print(meta.source_dataset.value_counts())
print("\nLabel distribution:")
print(meta.label.value_counts().rename({0: "benign", 1: "melanoma"}))
meta.head()

## Step 3: Visualize the Three Processing Modes

The `DermoscopyImageProcessor` can apply three modes using segmentation masks:
- **whole**: full image (no masking)
- **lesion**: only the lesion (background zeroed out)
- **background**: only the skin/background (lesion zeroed out)

This lets you test whether a classifier is learning from the lesion itself or from surrounding skin artifacts.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pyhealth.processors import DermoscopyImageProcessor

# Pick a sample row from metadata to visualize
sample_row = meta.iloc[0]
img_path  = sample_row["image_path"]
mask_path = sample_row["mask_path"]
label_str = "melanoma" if sample_row["label"] == 1 else "benign"
source    = sample_row["source_dataset"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(f"Source: {source} | Label: {label_str}", fontsize=13)

for ax, mode in zip(axes, ["whole", "lesion", "background"]):
    proc = DermoscopyImageProcessor(mode=mode, normalize=False)
    tensor = proc.process((img_path, mask_path))  # (3, H, W) in [0, 1]
    img_np = tensor.permute(1, 2, 0).numpy()       # (H, W, 3)
    ax.imshow(img_np)
    ax.set_title(f'mode="{mode}"', fontsize=12)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Step 4: Set Task and Build DataLoaders

The `DermoscopyMelanomaClassification` task maps each image+mask pair to a binary label.
Passing a custom `DermoscopyImageProcessor` allows you to switch modes.

In [ ]:
from pyhealth.tasks import DermoscopyMelanomaClassification
from pyhealth.processors import DermoscopyImageProcessor
from pyhealth.datasets import get_dataloader, split_by_sample

task = DermoscopyMelanomaClassification()

# Choose the processing mode — swap this to "lesion" or "background" to compare
image_processor = DermoscopyImageProcessor(
    mode=MODE,
    image_size=224,
    normalize=True,   # ImageNet normalization
)

samples = base_dataset.set_task(
    task=task,
    input_processors={"image": image_processor},
)

print(f"Total samples: {len(samples)}")
print(f"Input schema:  {samples.input_schema}")
print(f"Output schema: {samples.output_schema}")

In [ ]:
train_dataset, val_dataset, test_dataset = split_by_sample(samples, [0.7, 0.1, 0.2])

train_loader = get_dataloader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = get_dataloader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = get_dataloader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

## Step 5: Define the Model

We use `TorchvisionModel` with ResNet50 pretrained on ImageNet — the same architecture used in the
`dermatology_melanoma_classification.py` script from the dermoscopic_artifacts repo.

You can swap `model_name` to any supported architecture:
- `"resnet50"` (default here)
- `"densenet121"` 
- `"vit_b_16"` (Vision Transformer — recommended for ablation study)

In [ ]:
from pyhealth.models import TorchvisionModel

model = TorchvisionModel(
    dataset=samples,
    model_name="resnet50",
    model_config={"weights": "DEFAULT"},   # ImageNet pretrained weights
)

print(model)

## Step 6: Train the Model

In [ ]:
from pyhealth.trainer import Trainer

trainer = Trainer(
    model=model,
    device=DEVICE,
)

trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=NUM_EPOCHS,
    optimizer_params={"lr": LEARNING_RATE},
    monitor="roc_auc",
)

## Step 7: Evaluate the Model

In [ ]:
results = trainer.evaluate(test_loader)
print(f"\n=== Test results (mode='{MODE}') ===")
for metric, value in results.items():
    print(f"  {metric}: {value:.4f}")

## Step 8: Compare Modes (Artifact Bias Analysis)

The core research question is: **is the model relying on the lesion or on background artifacts?**

The cell below trains separate models for each mode and compares their AUC. If `background` AUC is
unexpectedly high, it suggests the model has learned spurious correlations from skin artifacts.

> ⚠️ This cell trains 3 full models and may take a while. Reduce `NUM_EPOCHS` or run one mode at a time.

In [ ]:
from pyhealth.models import TorchvisionModel
from pyhealth.trainer import Trainer
from pyhealth.datasets import get_dataloader, split_by_sample
from pyhealth.processors import DermoscopyImageProcessor

mode_results = {}

for mode in ["whole", "lesion", "background"]:
    print(f"\n{'='*50}")
    print(f" Training with mode='{mode}'")
    print(f"{'='*50}")

    # Re-process with this mode's processor
    mode_samples = base_dataset.set_task(
        task=DermoscopyMelanomaClassification(),
        input_processors={"image": DermoscopyImageProcessor(mode=mode)},
    )

    tr, va, te = split_by_sample(mode_samples, [0.7, 0.1, 0.2])
    tr_l = get_dataloader(tr, batch_size=BATCH_SIZE, shuffle=True)
    va_l = get_dataloader(va, batch_size=BATCH_SIZE, shuffle=False)
    te_l = get_dataloader(te, batch_size=BATCH_SIZE, shuffle=False)

    m = TorchvisionModel(
        dataset=mode_samples,
        model_name="resnet50",
        model_config={"weights": "DEFAULT"},
    )

    t = Trainer(model=m, device=DEVICE)
    t.train(tr_l, va_l, epochs=NUM_EPOCHS,
            optimizer_params={"lr": LEARNING_RATE}, monitor="roc_auc")

    mode_results[mode] = t.evaluate(te_l)

print("\n" + "="*50)
print(" Mode comparison summary")
print("="*50)
print(f"{'Mode':<15} {'ROC-AUC':>10} {'PR-AUC':>10} {'F1':>8}")
print("-"*45)
for mode, res in mode_results.items():
    roc = res.get("roc_auc", float("nan"))
    pr  = res.get("pr_auc",  float("nan"))
    f1  = res.get("f1",       float("nan"))
    print(f"{mode:<15} {roc:>10.4f} {pr:>10.4f} {f1:>8.4f}")

## Step 9: Per-Dataset Breakdown (Optional)

Run evaluation separately on each source dataset to check cross-dataset generalization.

In [ ]:
# Re-use the model trained in Step 6 (mode=MODE)
# Filter test set by source_dataset and evaluate each separately

import torch
from torch.utils.data import Subset

# Read back the metadata to get source labels for each test index
meta_reloaded = pd.read_csv(os.path.join(ROOT, "dermoscopy-metadata-pyhealth.csv"))

# The split preserves the original ordering, so we can align by index
# NOTE: this assumes split_by_sample uses row-index-aligned splits
print("Evaluating per source dataset on the test split...")
print("(Rerun Step 6 & 7 first if you haven't already)\n")

# Full test eval for reference
full_results = trainer.evaluate(test_loader)
print(f"Full test set  | ROC-AUC: {full_results.get('roc_auc', 'N/A'):.4f}")

---

## Summary

| Step | What happened |
|------|---------------|
| 1 | Configured data paths to ISIC 2018, HAM10000, and PH2 directories |
| 2 | Loaded `DermoscopyDataset` → unified `dermoscopy-metadata-pyhealth.csv` |
| 3 | Visualized 3 modes (whole / lesion / background) using segmentation masks |
| 4 | Applied `DermoscopyMelanomaClassification` task + `DermoscopyImageProcessor` |
| 5 | Initialized pretrained ResNet50 via `TorchvisionModel` |
| 6 | Trained with Adam + lr=1e-4, monitoring ROC-AUC |
| 7 | Evaluated on held-out test set |
| 8 | Compared all 3 modes to measure artifact bias |

### Key classes introduced

| Class | Location | Purpose |
|-------|----------|---------|
| `DermoscopyDataset` | `pyhealth.datasets` | Composite dataset (ISIC 2018 + HAM10000 + PH2) |
| `DermoscopyMelanomaClassification` | `pyhealth.tasks` | Binary melanoma task; returns (image_path, mask_path) tuples |
| `DermoscopyImageProcessor` | `pyhealth.processors` | Mode-based image processor; handles whole / lesion / background |

### Swap the model for ablation study

```python
# DenseNet121
model = TorchvisionModel(dataset=samples, model_name="densenet121", model_config={"weights": "DEFAULT"})

# Vision Transformer (ViT)
model = TorchvisionModel(dataset=samples, model_name="vit_b_16", model_config={"weights": "DEFAULT"})
```